# 1. Data Loading
---
This notebook is intended to be run both locally and on kaggle. Therefore, it needs accomodates the different data sources when loading the csvs. Before handling the data, imports will all be coalesced into a specified subsection.



## 1.1 Imports


In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder


## 1.2 Data Loading

In [ ]:
# 1. Automatically detect the environment
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    # Settings optimized for Kaggle's NVIDIA GPUs
    TREE_METHOD = "hist"  # XGBoost
    DEVICE_TYPE = "gpu"  # LightGBM (Kaggle uses OpenCL 'gpu', not 'cuda')
    CAT_TASK_TYPE = "GPU"  # CatBoost
    print("Environment: Kaggle (NVIDIA GPU Enabled)")
else:
    # Settings optimized for the local environment
    TREE_METHOD = "hist"  # XGBoost (Uses Metal/MPS natively via hist)
    DEVICE_TYPE = "cpu"  # LightGBM (Standard Mac wheels lack GPU support)
    CAT_TASK_TYPE = "CPU"  # CatBoost (Mac GPU support is experimental; CPU is faster)
    print("Environment: Local Mac (CPU fallback for LightGBM/CatBoost)")

Environment: Local Mac (CPU fallback for LightGBM/CatBoost)


In [ ]:
def load_data():
    """
    try to load the data from the local directory, if not found, load it from the Kaggle dataset.
    Output the three dataframes: train, test, and submission.
    """
    train_file_dir_name = "/train.csv"
    sample_submission_file_dir_name = "/sample_submission.csv"
    test_file_dir_name = "/test.csv"

    if IS_KAGGLE:
        # check if the data files are in kaggle direcotry
        # /kaggle/input/competitions/playground-series-s6e9
        kaggle_input_dir = "/kaggle/input/competitions/"
        competition_name = os.listdir(kaggle_input_dir)

        print("Trying to load data from kaggle input directory...")
        data_dir = "/kaggle/input/competitions/playground-series-s6e9/"
        train = pd.read_csv(
            kaggle_input_dir + competition_name[-1] + train_file_dir_name
        )
        test = pd.read_csv(kaggle_input_dir + competition_name[-1] + test_file_dir_name)
        submission = pd.read_csv(
            kaggle_input_dir + competition_name[-1] + sample_submission_file_dir_name
        )
        print("Data loaded successfully from kaggle input directory.")
    else:
        print("Trying to load data from local directory...")
        # Load data from local directory
        data_dir = os.getcwd() + "/data"
        train = pd.read_csv(data_dir + train_file_dir_name)
        test = pd.read_csv(data_dir + test_file_dir_name)
        submission = pd.read_csv(data_dir + sample_submission_file_dir_name)
        print("Data loaded successfully from local directory.")

    return train, test, submission

In [ ]:
train_df, test_df, submission_df = load_data()

Trying to load data from local directory...
Data loaded successfully from local directory.


# 2. Data exploration
---
Having organised the data into dataframes that can be explored and prepared for preprocessing, the necessary quality checks can be done. In this section, we will explore the data for missing values, extreme values and mislabelled data types. 